# 03 — PROMETHEE II ranking (real rule pools)

**Section:** Evaluation layer
**Depends on:** `artifacts/rules/{ga,nsga}_{baseline,sam}.json` (real extraction, 572 features, per-drug labels), `data/processed/real_extraction/`
**Feeds into:** `04_statistical_testing.ipynb`, `05_gsea_enrichment.ipynb`

This replaces the synthetic-fixture version. Changes from it, each deliberate:

| Item | Synthetic version | This version | Why |
|---|---|---|---|
| Data | `tests/fixtures/synthetic` | real pools + corrected extraction dataset | the real rules exist now |
| Fidelity criterion | value stored in the pool (train) | **re-scored on the held-out test split** with the rule's class fixed from train | train-for-search, test-for-reporting |
| Stability subgroup | `cancer_type` (6 subtypes) | **cell line** | no cancer-type / tissue annotation exists in the DeepCCDS precomputed files used by this project (it ships in Cell Model Passports, which was not downloaded) |
| Stability measure | variance of per-subgroup macro-F1 (`metrics.stability`) | variance of per-subgroup **agreement rate**; the contract version is reported alongside | see §3: macro-F1 of a one-class rule jumps from ~0.49 to 1.0 at perfect purity, which dominates the variance |
| Run | one PROMETHEE II over all rules | unchanged: **one** unified run over all 19 rules | φ values from separate runs are not comparable |

Criteria and weights follow methodology §4.6.1: fidelity 0.5 (maximise), stability 0.3 (minimise variance), comprehensibility 0.2 (minimise length). V-shape preference with `p` = observed range, `q = 0`, as in the synthetic notebook (still provisional).

In [1]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")            # run from the repo root so relative paths match the pipeline
sys.path.insert(0, ".")
import numpy as np, pandas as pd

from pymcdm.methods import PROMETHEE_II
from scipy.stats import spearmanr
from src.common import metrics
from src.evaluation import stage5_utils as U

pools, rule_lookup, feature_map = U.load_pools()
for n, p in pools.items():
    print(f"{n}: {len(p['rules'])} rules, runs={p['meta']['runs']}, config_hash={p['meta']['config_hash']}")
table = U.rule_table(pools)
print("\nall rules predict:", table.predicted_class.unique().tolist())
test = U.load_split("test"); train = U.load_split("train")
print("test rows:", len(test), "| cell lines:", test.cell_name.nunique(), "| drugs:", test.drug_name.nunique())

ga_baseline: 7 rules, runs=30, config_hash=a2070aedcc19
ga_sam: 6 rules, runs=30, config_hash=a2070aedcc19
nsga_baseline: 3 rules, runs=30, config_hash=a2070aedcc19
nsga_sam: 3 rules, runs=30, config_hash=a2070aedcc19

all rules predict: ['sensitive']


test rows: 30728 | cell lines: 866 | drugs: 379


## 1. Split structure (matters for how to read "held-out")

The corrected dataset splits **rows** (cell line × drug pairs) at random. Check which cell lines and drugs the splits share.

In [2]:
cells = {s: set(df.cell_name) for s, df in (("train", train), ("test", test))}
drugs = {s: set(df.drug_name) for s, df in (("train", train), ("test", test))}
print("cell lines in test that also appear in train:", len(cells["test"] & cells["train"]), "of", len(cells["test"]))
print("drugs in test that also appear in train:     ", len(drugs["test"] & drugs["train"]), "of", len(drugs["test"]))

cell lines in test that also appear in train: 866 of 866
drugs in test that also appear in train:      379 of 379


Every test cell line and drug also appears in train. Rules read cell-line features only, so test fidelity measures agreement with the model on **unseen (cell line, drug) pairs for cell lines the search has already seen**. It is a check against overfitting to specific rows, not a test on new cell lines. This goes in the limitations.

## 2. Re-score every rule on the test split

The rule's class is fixed (chosen by majority vote on train). `fidelity` compares it with the model's `class_pred`; `accuracy` compares it with the true per-drug class.

In [3]:
rows = []
for _, t in table.iterrows():
    rule = rule_lookup[t["id"]]
    s, _ = U.score_on_split(rule, test, t["condition"])
    rows.append({"id": t["id"], "fidelity_test": s["fidelity"], "accuracy_test": s["accuracy"], "coverage_test": s["coverage"],
                 "n_test_rows": s["n_rows"], "n_test_cells": s["n_cells"],
                 "model_agreement_test": s["share_pred_class_model"], "truth_agreement_test": s["share_pred_class_true"]})
table = table.merge(pd.DataFrame(rows), on="id")
table[["id", "length", "fidelity_train", "fidelity_test", "coverage_train", "coverage_test", "n_test_cells",
       "model_agreement_test", "truth_agreement_test", "accuracy_test"]].round(4)

,id,length,fidelity_train,fidelity_test,coverage_train,coverage_test,n_test_cells,model_agreement_test,truth_agreement_test,accuracy_test
0,ga_baseline_0001,3,0.4800,0.4780,0.0221,0.0216,18,0.9155,0.8220,0.4512
1,ga_baseline_0002,2,0.4793,0.4776,0.0245,0.0242,20,0.9141,0.8121,0.4481
2,ga_baseline_0003,2,0.4788,0.4788,0.0240,0.0248,22,0.9187,0.8231,0.4515
3,ga_baseline_0004,4,0.4774,0.4796,0.0290,0.0299,27,0.9217,0.8467,0.4585
4,ga_baseline_0005,3,0.4761,0.4755,0.0289,0.0300,26,0.9066,0.8274,0.4528
5,ga_baseline_0006,1,0.4640,0.4657,0.1427,0.1455,126,0.8716,0.7945,0.4427
6,ga_baseline_0007,2,0.4605,0.4599,0.0254,0.0252,22,0.8516,0.7987,0.4440
7,ga_sam_0001,4,0.4914,0.4938,0.0238,0.0225,21,0.9754,0.8786,0.4677
8,ga_sam_0002,3,0.4913,0.4917,0.0241,0.0239,21,0.9673,0.8542,0.4607
9,ga_sam_0003,7,0.4861,0.4882,0.0247,0.0240,22,0.9540,0.8633,0.4633


**Reading.**
- Train and test fidelity agree to about 0.005, as expected when test shares cell lines with train.
- `model_agreement_test` is the share of covered test rows where the model says the rule's class, and is what fidelity encodes (`fidelity = p/(1+p)`). Rules agree with the model on roughly 79–98% of what they cover.
- `truth_agreement_test` is agreement with the true per-drug label: lower (about 73–88%), because the model is not perfect; the rule explains the model, not the data.
- Rules sit at coverage of about 2–3% (or 14–18% for one-condition rules). The 2% floor is **binding**: the search pushes coverage down to the floor to buy purity. Fidelity is therefore conditional on the floor value (future work: sweep it).
- At the floor a rule covers only about 18–27 distinct cell lines (`n_test_cells`) out of 866, so "2% of rows" is closer to "20 cell lines, each with many drugs". Rows within a cell line are not independent.

## 3. Stability

Subgroup = cell line (≥ 10 covered test rows). For each subgroup, the **agreement rate** is the share of its covered rows on which the model's class equals the rule's class; stability is the variance of that across subgroups (lower = more stable). The contract definition (`metrics.stability`, variance of per-subgroup macro-F1) is computed too, to show why it is not used for ranking.

In [4]:
rows = []
for _, t in table.iterrows():
    rule = rule_lookup[t["id"]]
    va, ng, per = U.stability_agreement(rule, test, t["condition"])
    vf, _, perf = U.rule_stability(rule, test, t["condition"])
    rows.append({"id": t["id"], "stability": va, "stability_contract_f1": vf, "n_stability_groups": ng,
                 "min_group_agreement": min(per), "share_groups_perfectly_pure": float(np.mean(np.array(per) == 1.0))})
table = table.merge(pd.DataFrame(rows), on="id")
table[["id", "condition", "model_agreement_test", "stability", "stability_contract_f1", "n_stability_groups",
       "share_groups_perfectly_pure"]].round(4)

,id,condition,model_agreement_test,stability,stability_contract_f1,n_stability_groups,share_groups_perfectly_pure
0,ga_baseline_0001,baseline,0.9155,0.0054,0.0568,18,0.2778
1,ga_baseline_0002,baseline,0.9141,0.0051,0.0530,20,0.2500
2,ga_baseline_0003,baseline,0.9187,0.0052,0.0490,22,0.2273
3,ga_baseline_0004,baseline,0.9217,0.0099,0.0429,27,0.1852
4,ga_baseline_0005,baseline,0.9066,0.0159,0.0623,26,0.3077
5,ga_baseline_0006,baseline,0.8716,0.0226,0.0497,126,0.1984
6,ga_baseline_0007,baseline,0.8516,0.0368,0.0296,22,0.0909
7,ga_sam_0001,sam,0.9754,0.0017,0.0650,21,0.5714
8,ga_sam_0002,sam,0.9673,0.0018,0.0665,21,0.4762
9,ga_sam_0003,sam,0.9540,0.0055,0.0679,22,0.4545


In [5]:
# Does the contract version track model agreement, or "how many subgroups happen to be perfectly pure"?
print("Spearman(contract stability, share of perfectly pure subgroups):",
      round(spearmanr(table.stability_contract_f1, table.share_groups_perfectly_pure)[0], 2))
print("Spearman(smooth stability,   share of perfectly pure subgroups):",
      round(spearmanr(table.stability, table.share_groups_perfectly_pure)[0], 2))
print("\nContract stability, mean by model condition (GA rules only):")
print(table[table.extractor == "ga"].groupby("condition")[["model_agreement_test", "stability", "stability_contract_f1"]].mean().round(4))

Spearman(contract stability, share of perfectly pure subgroups): 0.8
Spearman(smooth stability,   share of perfectly pure subgroups): -0.46

Contract stability, mean by model condition (GA rules only):
           model_agreement_test  stability  stability_contract_f1
condition                                                        
baseline                  0.900     0.0144                  0.049
sam                       0.941     0.0101                  0.068


In the contract version the SAM GA rules, which agree most with the model, receive the **worst** stability, because their subgroups are more often perfectly pure and a pure subgroup scores 1.0 against about 0.49 for a 97%-pure one. That is an artefact of macro-F1 on one-class rules, not instability. The smooth version is used for the ranking; the contract version is carried through the sensitivity analysis in §6.

## 4. Are the conditions informative? (drop-one check, train rows)

`retention = coverage(rule) / coverage(rule without that condition)`. A retention near 1.0 would mean the condition removes nothing.

In [6]:
vac = []
for rid in table.id:
    for v in U.condition_vacuity(rule_lookup[rid], train):
        vac.append({"id": rid, **v})
vac = pd.DataFrame(vac)
vac["is_mutation"] = vac.feature.str.endswith("_mut")
print("conditions in total:", len(vac), "| mutation-side:", int(vac.is_mutation.sum()), "| near-vacuous (retention >= 0.95):", int(vac.vacuous.sum()))
print(vac[vac.vacuous][["id", "feature", "partition", "retention"]].round(3).to_string(index=False))
print("\nmutation 'Low' (= not mutated) conditions:", int(((vac.partition == "Low") & vac.is_mutation).sum()),
      "| median retention:", round(vac[(vac.partition == "Low") & vac.is_mutation].retention.median(), 3))
vac.to_csv(U.REPORT_DIR / "condition_drop_one_retention.csv", index=False)

conditions in total: 49 | mutation-side: 25 | near-vacuous (retention >= 0.95): 2
         id    feature partition  retention
ga_sam_0002 NKX2-1_mut       Low      0.952
ga_sam_0003  GNA11_mut       Low      1.000

mutation 'Low' (= not mutated) conditions: 14 | median retention: 0.747


Nearly all conditions do remove rows: even "not mutated" conditions typically cut 5–50% of the region, i.e. they refine which cell lines are covered. Only a couple are near-vacuous, so `length` is a fair comprehensibility proxy here.

## 5. PROMETHEE II, one unified run over all 19 rules

In [7]:
CRITERIA = ["fidelity_test", "length", "stability"]
WEIGHTS = np.array([0.5, 0.2, 0.3])          # fidelity, comprehensibility, stability
TYPES = np.array([1, -1, -1])                # 1 = maximise, -1 = minimise

def run_promethee(df, criteria, weights):
    m = df[criteria].to_numpy(float)
    assert not np.isnan(m).any()
    p = m.max(axis=0) - m.min(axis=0); q = np.zeros(len(criteria))
    return PROMETHEE_II("vshape", p=p, q=q)(m, weights, TYPES), p, q

phi, p, q = run_promethee(table, CRITERIA, WEIGHTS)

def phi_manual(m, w, types, p):
    n = len(m); pi = np.zeros((n, n))
    for j in range(m.shape[1]):
        d = (m[:, None, j] - m[None, :, j]) * types[j]
        pi += w[j] * np.clip(d / p[j], 0, 1)
    return (pi.sum(1) - pi.sum(0)) / (n - 1)

chk = phi_manual(table[CRITERIA].to_numpy(float), WEIGHTS, TYPES, p)
print("pymcdm vs independent numpy implementation, max |difference|:", float(np.abs(phi - chk).max()))
assert np.allclose(phi, chk)
table["phi"] = phi
ranked = table.sort_values("phi", ascending=False).reset_index(drop=True)
ranked.index += 1
print("sum of phi (must be ~0):", round(float(phi.sum()), 12))
ranked[["id", "extractor", "condition", "fidelity_test", "length", "stability", "coverage_test", "phi"]].round(4)

pymcdm vs independent numpy implementation, max |difference|: 5.551115123125783e-17
sum of phi (must be ~0): 0.0


,id,extractor,condition,fidelity_test,length,stability,coverage_test,phi
1,ga_sam_0002,ga,sam,0.4917,3,0.0018,0.0239,0.2776
2,ga_sam_0001,ga,sam,0.4938,4,0.0017,0.0225,0.2631
3,ga_sam_0004,ga,sam,0.4863,2,0.0057,0.0488,0.2455
4,ga_baseline_0003,ga,baseline,0.4788,2,0.0052,0.0248,0.1740
5,ga_baseline_0002,ga,baseline,0.4776,2,0.0051,0.0242,0.1619
6,ga_sam_0005,ga,sam,0.4794,2,0.0124,0.0419,0.1525
7,ga_baseline_0001,ga,baseline,0.4780,3,0.0054,0.0216,0.1296
8,ga_baseline_0004,ga,baseline,0.4796,4,0.0099,0.0299,0.0942
9,ga_sam_0003,ga,sam,0.4882,7,0.0055,0.0240,0.0892
10,ga_baseline_0005,ga,baseline,0.4755,3,0.0159,0.0300,0.0670


In [8]:
# top-10 per (extractor, condition) pool = every rule (pools have 3-7 rules)
top = ranked.groupby(["extractor", "condition"]).head(10)
print(top.groupby(["extractor", "condition"]).size())
print("\nBest rule per pool:")
print(ranked.groupby(["extractor", "condition"]).head(1)[["id", "fidelity_test", "length", "stability", "phi"]].round(4).to_string())
print("\nMean phi by pool:")
print(ranked.groupby(["extractor", "condition"]).phi.agg(["mean", "median", "count"]).round(3))

extractor  condition
ga         baseline     7
           sam          6
nsga       baseline     3
           sam          3
dtype: int64

Best rule per pool:
                    id  fidelity_test  length  stability     phi
1          ga_sam_0002         0.4917       3     0.0018  0.2776
4     ga_baseline_0003         0.4788       2     0.0052  0.1740
13  nsga_baseline_0001         0.4643       3     0.0225 -0.0674
17       nsga_sam_0001         0.4545       4     0.0648 -0.3542

Mean phi by pool:
                      mean  median  count
extractor condition                      
ga        baseline   0.074   0.094      7
          sam        0.172   0.199      6
nsga      baseline  -0.117  -0.121      3
          sam       -0.399  -0.367      3


## 6. Sensitivity (supplementary): does the ranking depend on the provisional weights, or on the stability definition?

In [9]:
variants = {
    "base (0.5/0.2/0.3)": (CRITERIA, WEIGHTS),
    "equal weights":        (CRITERIA, np.array([1/3, 1/3, 1/3])),
    "fidelity-heavy":       (CRITERIA, np.array([0.7, 0.1, 0.2])),
    "stability-heavy":      (CRITERIA, np.array([0.3, 0.2, 0.5])),
    "length-heavy":         (CRITERIA, np.array([0.3, 0.4, 0.3])),
    "contract stability":   (["fidelity_test", "length", "stability_contract_f1"], WEIGHTS),
    "train fidelity":       (["fidelity_train", "length", "stability"], WEIGHTS),
}
sens = pd.DataFrame({"id": table.id})
for name, (crit, w) in variants.items():
    sens[name] = run_promethee(table, crit, w)[0]
base_rank = sens["base (0.5/0.2/0.3)"].rank(ascending=False)
out = []
for name in variants:
    ph = sens[name]
    top3 = list(sens.loc[ph.sort_values(ascending=False).index[:3], "id"])
    out.append({"variant": name, "Spearman vs base": round(spearmanr(base_rank, ph.rank(ascending=False))[0], 3), "top 3": top3})
pd.DataFrame(out)

,variant,Spearman vs base,top 3
0,base (0.5/0.2/0.3),1.000,"[ga_sam_0002, ga_sam_0001, ga_sam_0004]"
1,equal weights,0.949,"[ga_sam_0004, ga_sam_0002, ga_baseline_0003]"
2,fidelity-heavy,0.968,"[ga_sam_0001, ga_sam_0002, ga_sam_0004]"
3,stability-heavy,0.995,"[ga_sam_0002, ga_sam_0004, ga_sam_0001]"
4,length-heavy,0.884,"[ga_sam_0004, ga_sam_0002, ga_baseline_0003]"
5,contract stability,0.935,"[ga_sam_0002, ga_sam_0001, ga_baseline_0003]"
6,train fidelity,0.988,"[ga_sam_0002, ga_sam_0001, ga_sam_0004]"


In [10]:
# mean phi by condition under each variant (does the SAM-vs-baseline picture change?)
tmp = sens.merge(table[["id", "extractor", "condition"]], on="id")
tmp.groupby(["extractor", "condition"])[list(variants)].mean().round(3)

base (0.5/0.2/0.3)  equal weights  fidelity-heavy  \
extractor condition                                                      
ga        baseline                0.074          0.072           0.069   
          sam                     0.172          0.119           0.219   
nsga      baseline               -0.117         -0.071          -0.161   
          sam                    -0.399         -0.335          -0.438   

                     stability-heavy  length-heavy  contract stability  \
extractor condition                                                      
ga        baseline             0.089         0.067               0.088   
          sam                  0.154         0.097               0.083   
nsga      baseline            -0.093        -0.054              -0.036   
          sam                 -0.422        -0.297              -0.333   

                     train fidelity  
extractor condition                  
ga        baseline            0.086  
          sam                 0.173  
nsga      baseline           -0.150  
          sam                -0.397

## 7. One matched comparison: the same rule under both models

A rule with identical conditions found under both baseline and SAM is the cleanest like-for-like comparison available (same region of cell lines, different model).

In [11]:
sig = {rid: tuple(sorted((c["feature"], c["partition"]) for c in rule_lookup[rid]["conditions"])) for rid in table.id}
table["_sig"] = table.id.map(sig)
dup = table[table.duplicated("_sig", keep=False)].sort_values("_sig")
dup[["id", "condition", "fidelity_test", "model_agreement_test", "stability", "coverage_test"]].round(4) if len(dup) else print("no identical rules across conditions")

,id,condition,fidelity_test,model_agreement_test,stability,coverage_test
5,ga_baseline_0006,baseline,0.4657,0.8716,0.0226,0.1455
12,ga_sam_0006,sam,0.4686,0.8819,0.0334,0.1455


## 8. Save

In [12]:
U.REPORT_DIR.mkdir(parents=True, exist_ok=True)
save = ranked.drop(columns=["_sig"], errors="ignore")
save.to_csv(U.REPORT_DIR / "promethee_ranked_rules.csv", index=False)
top.drop(columns=["_sig"], errors="ignore").to_csv(U.REPORT_DIR / "promethee_top10_per_condition.csv", index=False)
sens.to_csv(U.REPORT_DIR / "promethee_sensitivity.csv", index=False)
json.dump({"criteria": CRITERIA, "weights": WEIGHTS.tolist(), "types": TYPES.tolist(), "preference_function": "vshape",
           "p": p.tolist(), "q": q.tolist(), "config_hash": pools["ga_baseline"]["meta"]["config_hash"],
           "stability": "variance across cell lines (>=10 covered test rows) of per-cell-line agreement rate",
           "fidelity": "macro-F1 vs model class_pred on the TEST split, rule class fixed from train",
           "status": "weights and preference function provisional (not signed off by supervisors)"},
          open(U.REPORT_DIR / "promethee_config.json", "w"), indent=2)
print("saved to", U.REPORT_DIR)

saved to artifacts/reports


## Limitations specific to this notebook

- All 19 rules predict "sensitive"; no "resistant" rule exists, so the pools describe only the sensitive side of the models.
- Stability uses cell lines because tissue / cancer type is unavailable; it is a proxy for the §4.6.1 subtype criterion, not the same quantity.
- Weights and the V-shape preference function are provisional; §6 shows how much they matter.
- Test shares cell lines and drugs with train (§1), so test fidelity does not measure generalisation to new cell lines.
- Rules are global and drug-blind (they read cell-line features only), which caps how pure a covered region can be.